# Can a computer sort the ocean into its water masses?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/11_water_masses.ipynb).

Lower a bottle into the Atlantic and bring it up: the water inside sank from the surface years or
centuries ago, far away, and still carries the temperature and saltiness it had there. Oceanographers
call such a body of water a **water mass**, and have named the main ones from thousands of bottles.

Today a computer tries the same with no names to learn from: two numbers, temperature and salinity, for
6,123 places on one north–south slice of the Atlantic. You sort them with **k-means**,
**hierarchical clustering** and **DBSCAN**, check the groups against the oceanographers' water masses, choose
each method's settings, and find where each water mass was made. At home, the Pacific.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** What a water mass is; the three that fill the deep Atlantic, where each sits and where
each was made; why the deep Pacific has no salty layer.

**The code.** A distance with a scaling factor for each column · scale and group in one
`make_pipeline(StandardScaler(), method)`, then check the groups · `KMeans`,
`AgglomerativeClustering` and `DBSCAN` · choosing their hyperparameters: k, eps and min_samples.

**Thirteen places where you write something: nine in class, four at home.** Each is headed *Your turn*, and the cell under it has the shape of the answer written
in, with `...` wherever your code goes and a comment beside each one saying what goes there. Early on the
comment says how; later, once you have seen a thing done, it says only what.

The eight questions of the class:

1. What fills the Atlantic, from the surface to the sea floor?
2. How alike are two samples of water?
3. How can a computer sort the samples into groups?
4. Are the groups the water masses oceanographers name?
5. How many groups?
6. Does a second method find the same groups?
7. Where does the saltiest water spread at 1000 m?
8. Where was each water mass made?

## Setup

Run the next cell once. It reads three tables from the **World Ocean Atlas 2023** (NOAA), decades of
ship and float measurements averaged into one value per place and depth:

- `atlantic`: a slice at 25.5°W, from the Weddell Sea (70.5°S) to between Greenland and Iceland (65.5°N),
  every 100 m down to 5500 m: `lat` (°N; south is negative), `depth_m`, `temperature_c` and `salinity`.
- `water_1000m`: every 1° square of ocean at 1000 m, with `lat`, `lon` (°E; west is negative),
  `temperature_c` and `salinity` (the Black Sea, far fresher than any other, left out).
- `pacific`: the same kind of slice through the Pacific at 150.5°W, for the homework.

It also defines two functions, `coldest_first` and `match_colours` (section 3).

**Salinity.** How salty seawater is, close to grams of salt in a kilogram of water: 35 means 3.5 percent salt.

**Potential temperature.** The temperature a water sample would have if it were lifted to the surface; deep water is squeezed by the water above and warmed a little, so oceanographers remove that warming before comparing waters. The column `temperature_c` holds it, in °C.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

# World Ocean Atlas 2023, NOAA (Locarnini et al. 2024; Reagan et al. 2024): yearly means over decades
atlantic = pd.read_csv(CACHE + "/week11w_atlantic_25w.csv")        # the Atlantic slice at 25.5°W
water_1000m = pd.read_csv(CACHE + "/week11w_global_1000m.csv")     # every 1° square of ocean, at 1000 m
pacific = pd.read_csv(CACHE + "/week11w_pacific_150w.csv")         # the Pacific slice at 150.5°W, for the homework
coast = pd.read_csv(CACHE + "/coastlines.csv")
ts_columns = ["temperature_c", "salinity"]                       # the only two numbers every method sees


def coldest_first(table, labels):
    """the same groups, renumbered by median temperature: 0 for the coldest group (section 3)"""
    medians = table.groupby(labels)["temperature_c"].median()   # groupby also takes an array: one group number per row
    renumbered = np.array(labels)                               # a new array holding the same numbers, to renumber
    place = 0
    for group in medians.sort_values().index:                  # the group numbers, coldest first
        renumbered[labels == group] = place
        place = place + 1
    return renumbered


def match_colours(labels, reference):
    """the same groups, renumbered so each takes the number, and so the colour, of the reference group
    that holds most of its samples; a group with no such number free gets a new one (sections 3 and 7)"""
    renumbered = np.array(labels)
    taken = []
    new_number = np.max(reference) + 1                          # the first number no reference group uses
    for group in pd.Series(labels).value_counts().index:        # the groups, largest first
        if group == -1:                                         # DBSCAN's noise stays noise
            continue
        matches = reference[(labels == group) & (reference >= 0)]   # its samples' reference groups, noise left out
        if len(matches) > 0 and pd.Series(matches).value_counts().index[0] not in taken:
            best = pd.Series(matches).value_counts().index[0]   # the reference group holding most of its samples
        else:                                                   # none, or given to a bigger group: a new number
            best = new_number
            new_number = new_number + 1
        renumbered[labels == group] = best
        taken.append(best)
    return renumbered

print(len(atlantic), "Atlantic samples;", len(water_1000m), "squares at 1000 m;", len(pacific), "Pacific samples")

## 1. What fills the Atlantic, from the surface to the sea floor?

**The ocean is layered, not stirred.** Water sinks from the surface only in a few places, where it is
cold or salty enough to be heavier than the water below, then spreads at depth for thousands of
kilometres, mixing slowly. It keeps the temperature and salinity it took at the surface: a **water mass**.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/talley_slide_formation_sites.png" width="720">

*Where surface water sinks. X: the water is cooled or made salty until it is heavier than the water below, and it sinks. B: sea ice forms and leaves its salt behind in the water below it. From L. Talley's oceanography lectures (UC San Diego, SIO 210, 2018).*

The Labrador and Greenland Seas make the deep water of the North Atlantic; the Weddell and Ross Seas the
bottom water around Antarctica; the Mediterranean and the Red Sea salty water of their own (section 7).
The map shows today's slice, and dashed, the homework's.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.6)
plt.plot([-25.5, -25.5], [np.min(atlantic["lat"]), np.max(atlantic["lat"])], color="tab:blue", lw=3, label="class: the Atlantic at 25.5°W")
plt.plot([-150.5, -150.5], [np.min(pacific["lat"]), np.max(pacific["lat"])], color="tab:orange", lw=2, ls="--", label="homework: the Pacific at 150.5°W")
plt.gca().set_aspect("equal")
plt.xlabel("longitude (°E)")
plt.ylabel("latitude (°N)")
plt.title("The two slices through the ocean")
plt.legend(loc="lower left")
plt.show()

### ✏️ Your turn 1

**What do temperature and salinity look like on the slice?**

**Why:** before any method, look at the two columns every method will see, where they sit in the ocean.

1. **`temperature_points`**: the left panel, the slice coloured by temperature.
2. **`salinity_points`**: the right panel, coloured by salinity.

So: which column shows layers that are nearly flat, and which shows tongues of water that slope?

**Use these names**, because the self-check looks for them: `temperature_points` and `salinity_points`.

In [ ]:
# ← your answer here: replace every ... with code
# atlantic: the slice; one dot per sample, latitude across and depth down
# keep each scatter in a name, so the self-check can look at what you drew
plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)      # the left panel
temperature_points = ...  # plt.scatter: latitude across, depth on the y-axis, coloured (c=) by temperature; cmap="viridis", vmin=0, vmax=25, s=9, marker="s"
plt.colorbar(label="temperature (°C)")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("Temperature at 25.5°W")
plt.subplot(1, 2, 2)      # the right panel
salinity_points = ...     # the same scatter, coloured (c=) by salinity; cmap="viridis", vmin=34.0, vmax=35.2, s=9, marker="s"
plt.colorbar(label="salinity (the colours stop at 34.0 and 35.2)")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("Salinity at 25.5°W")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['temperature_points', 'salinity_points']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
for points in (temperature_points, salinity_points):
    assert len(points.get_offsets()) == len(atlantic), "each panel draws every row of atlantic: x and y are columns of atlantic"
    assert points.get_array() is not None, "colour each dot by a column: c=atlantic[...]"
    assert (points.get_offsets()[:, 0] == atlantic["lat"]).all() and (points.get_offsets()[:, 1] == atlantic["depth_m"]).all(), "in each panel, atlantic[\"lat\"] goes across and atlantic[\"depth_m\"] on the y-axis"
assert list(temperature_points.get_array()) == list(atlantic["temperature_c"]), "the left panel: c=atlantic[\"temperature_c\"]"
assert list(salinity_points.get_array()) == list(atlantic["salinity"]), "the right panel: c=atlantic[\"salinity\"]"
print(f"✓ Your turn 1 — the slice drawn by temperature and by salinity")

**Temperature is layered nearly flat:** warm only in the top few hundred metres, cold almost everywhere
below 1000 m, so it hardly tells one deep water from another. **Salinity shows what temperature hides:** a dark,
fresh tongue sinks from near 50°S to about 1 km; salty water from the north fills 1.5 to 4 km; a third,
darker water lies below 4 km in the south; and a very salty patch (yellow) at 30–40°N reaches below
1 km. (The white gap at 37.5°N is São Miguel, in the Azores.)

**Oceanographers name the layers:** five rows, the warm surface water and four layers below it, each
layer held mainly by one water mass on this slice (Liu and Tanhua, 2021, *Ocean Science*):

| Layer | Main water mass (and others in the layer) | Where it leaves the surface |
|---|---|---|
| surface | warm surface water | — (it is the surface, stirred by the wind) |
| upper | central water | the open ocean at middle latitudes, in winter |
| intermediate | **Antarctic Intermediate Water**: fresh (at 30–40°N, salty **Mediterranean Water**) | south of 40°S (the Mediterranean Sea, over the sill at Gibraltar) |
| deep | **North Atlantic Deep Water**: salty | the Labrador Sea and the seas between Greenland and Norway |
| bottom | **Antarctic Bottom Water**: the coldest (north of the equator, mixed on the way, it is called Northeast Atlantic Bottom Water) | around Antarctica |

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/lt2021_fig04.png" width="900">

*Left: salinity on the same line through the Atlantic (about 25°W), with the water masses named: AAIW, Antarctic Intermediate Water; MOW, Mediterranean Water; NADW, North Atlantic Deep Water; AABW, Antarctic Bottom Water. Right: the line on the map (blue, A16). Liu and Tanhua (2021), Ocean Science, Fig. 4, CC BY 4.0.*

**The same line, named:** the tongues of your salinity plot, labelled by oceanographers.

Water masses show best on a **T–S diagram**: each sample one dot, salinity across, temperature up. The
next cell draws one, coloured by depth.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(atlantic["salinity"], atlantic["temperature_c"], c=atlantic["depth_m"], cmap="viridis_r", s=4)   # viridis_r: viridis reversed, so the deep water is dark
plt.colorbar(label="depth (m)")
plt.xlabel("salinity")
plt.ylabel("temperature (°C)")
plt.title(f"T–S diagram of the Atlantic slice ({len(atlantic)} samples)")
plt.show()

**Each arm of the cloud is a water mass.** The warm waters run to the top right; below 5 °C three arms
meet: fresh to the left (Antarctic Intermediate Water), salty to the right (North Atlantic Deep Water),
coldest at the bottom (Antarctic Bottom Water). Where water masses meet they mix, so the arms join with
no gap.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/talley_slide_atlantic_ts.png" width="820">

*The textbook T–S diagram of the whole Atlantic, coloured by latitude, with arrows to each water mass. Talley and others (2011), Descriptive Physical Oceanography, Fig. 9.18, from L. Talley's SIO 210 lectures (2018).*

The textbook diagram of the whole Atlantic has the same arms, each named.

**Today's question:** from temperature and salinity alone, can a computer find these water masses? Its
groups are then checked against depth and latitude, which it never saw.

## 2. How alike are two samples of water?

Last week every basalt came with its label. No water sample comes with its water mass, so the computer
sorts the samples by their numbers alone: **clustering**. Whether the groups mean anything is checked
afterwards.

**Clustering puts samples that are near each other in one group, so it needs a distance.** Take one
station, 30.5°S:

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/ctd_rosette_ooi.jpg" width="520">

*A CTD rosette going over the side: 24 sampling bottles around a sensor that records temperature, salinity and depth all the way down. Photo: Ocean Observatories Initiative (public domain).*

Our numbers average decades of casts like this one. Three bottles from that station:

In [ ]:
three = atlantic[(atlantic["lat"] == -30.5) & atlantic["depth_m"].isin([800, 1000, 2000])]   # one station; .isin: depth is one of the three
print(three[["depth_m", "temperature_c", "salinity"]])

The 800 m and 1000 m bottles hold the same fresh Antarctic water (both salinity 34.32). The
2000 m bottle is close to the 1000 m bottle in temperature (2.63 against 3.77 °C) but saltier
(34.8): different water. A good distance should put the 1000 m bottle nearer the 800 m bottle.

**The distance between two samples** is measured as on a map: the difference in each column, squared,
added, and the square root taken (Pythagoras). Temperature is in °C and salinity has no unit, so each
difference is first divided by a **scaling factor**:

$$d = \sqrt{\left(\frac{T_1 - T_2}{s_T}\right)^2 + \left(\frac{S_1 - S_2}{s_S}\right)^2}$$

Here $T_1$ and $T_2$ are the temperatures of the two samples, $S_1$ and $S_2$ their salinities, $s_T$
and $s_S$ the scaling factors, and $d$ the distance.

Without scaling factors, 1 °C counts as much as 1 in salinity, yet 1 °C is a small change in the ocean and
1 in salinity a huge one, so temperature decides everything. The usual scaling factor is the column's standard deviation over the whole slice:
4.32 °C for temperature, only 0.36 for salinity. This is why last week's SVM needed
`StandardScaler`.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/scaling_mechanism.png" width="900">

*The bottles from 800 m down at 30.5°S, with a red line from the 1000 m bottle to its nearest other bottle. Left: raw units, 1 °C drawn as long as 1 unit of salinity. Right: each column divided by its scaling factor. Drawn for this course.*

**Scaled, the 1000 m bottle moves next to the 800 m bottle,** the same water.

## 3. How can a computer sort the samples into groups?

The first of three methods, and the most used, is **k-means**.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/kmeans_mechanism.png" width="900">

*How k-means works, on a made-up cloud of points. Drawn for this course.*

In words, with $k$ centres:

1. **Start**: put the $k$ centres somewhere.
2. **Assign**: give each sample to its nearest centre, by the distance $d$.
3. **Move**: put each centre at the mean of its samples ("means" gives the method its name).
4. **Repeat** 2 and 3 until no sample changes centre.

Each assign and each move can never increase one number: the sum, over all samples, of the squared distance
to their centre. k-means stops when it no longer shrinks. Where it ends depends on where its centres
start, so `n_init=10` restarts it ten times and keeps the run with the smallest sum, and `random_state=0`
makes everyone's run the same.

**Every clustering method takes the same three steps:**

1. **Scale** each column by its standard deviation (`StandardScaler`).
2. **Group**: run the method; it hands back one group number per row.
3. **Check** the groups against something the method never saw.

As last week, `make_pipeline(StandardScaler(), method)` joins steps 1 and 2 into one model, and
`.fit_predict(table)` runs both.

**Group numbers.** A method numbers its groups in whatever order it finds them, so the same water can come
back under another number and colour. The setup cell's `coldest_first(table, labels)` renumbers the groups
of Your turn 2 by median temperature: **blue the coldest, then orange, green, red, purple.** Later runs, with
another k or method, go through `match_colours(labels, groups)`: each group takes the colour of the Your
turn 2 group that holds most of its samples, and a group left over gets a new colour (brown, pink, ...).
Above k = 5 each panel matches the panel before. So in every k-means and hierarchical figure of the
Atlantic, the same water keeps its colour.

### ✏️ Your turn 2

**Sort the Atlantic slice into five groups with k-means.**

**Why:** first k-means on the columns as they are, then with a scaler in front, to see what scaling does.
Five groups, one per row of the layer table. Each model is made on one line and run on the next.

1. **`atlantic_k`**: the number of groups, 5. Every later cell uses this name, not the number.
2. **`kmeans`**: the k-means model, with `atlantic_k` groups.
3. **`raw_groups`**: run it on the slice's two columns, `atlantic[ts_columns]`, unscaled.
4. **`kmeans_pipe`**: a `StandardScaler()` and a new k-means with the same settings in a pipeline, like last week's
   `make_pipeline(StandardScaler(), SVC())`.
5. **`groups`**: run it on the same columns.

The given rest draws both groupings on the T–S diagram. So: which one follows the arms of the cloud?

**Use these names**, because the self-check looks for them: `atlantic_k`, `kmeans`, `raw_groups`,
`kmeans_pipe` and `groups`.

In [ ]:
# ← your answer here: replace every ... with code
# atlantic: the slice; ts_columns: its temperature and salinity columns
atlantic_k = ...                                            # how many groups: one per row of the layer table
# k-means on the columns as they are
kmeans = KMeans(n_clusters=..., n_init=10, random_state=0)  # make the model, with atlantic_k groups
raw_groups = kmeans.fit_predict(...)                        # run it on the slice's two columns

# a new k-means with the same settings, with a scaler in front
kmeans_pipe = make_pipeline(..., ...)  # step 0: a StandardScaler(); step 1: a new KMeans(...) with the same three settings
groups = kmeans_pipe.fit_predict(...)  # run it on the same two columns

groups = coldest_first(atlantic, groups)                                          # coldest_first, from setup: renumber, 0 for the coldest group
raw_groups = coldest_first(atlantic, raw_groups)
print("the scaling factors the scaler learned:", kmeans_pipe[0].scale_.round(2))  # step 0, the scaler; .scale_: the standard deviations it learned

plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1)
# cmap="tab10": ten clearly different colours; vmin=0, vmax=9 give group 0 the first and group 9 the last
plt.scatter(atlantic["salinity"], atlantic["temperature_c"], c=raw_groups, cmap="tab10", vmin=0, vmax=9, s=4)
plt.xlabel("salinity")
plt.ylabel("temperature (°C)")
plt.title("k-means, scaling factors 1 and 1")
plt.subplot(1, 2, 2)
plt.scatter(atlantic["salinity"], atlantic["temperature_c"], c=groups, cmap="tab10", vmin=0, vmax=9, s=4)
plt.xlabel("salinity")
plt.ylabel("temperature (°C)")
plt.title("k-means, scaled")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['atlantic_k', 'kmeans', 'raw_groups', 'kmeans_pipe', 'groups']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert atlantic_k == 5 and kmeans.n_clusters == atlantic_k and len(raw_groups) == len(atlantic), "kmeans = KMeans(n_clusters=5, n_init=10, random_state=0), then raw_groups = kmeans.fit_predict(atlantic[ts_columns])"
assert type(kmeans_pipe) == type(make_pipeline(StandardScaler())) and len(kmeans_pipe) == 2, "kmeans_pipe = make_pipeline(StandardScaler(), KMeans(...)): the scaler first"
assert kmeans.random_state == 0 and kmeans_pipe[1].random_state == 0 and kmeans.n_init == 10 and kmeans_pipe[1].n_init == 10, "both KMeans need n_init=10 and random_state=0, so everyone gets the same run"
assert len(kmeans_pipe) == 2 and list(kmeans_pipe[0].scale_.round(2)) == [4.32, 0.36], "kmeans_pipe = make_pipeline(StandardScaler(), KMeans(...)), then groups = kmeans_pipe.fit_predict(atlantic[ts_columns])"
assert list(pd.Series(groups).value_counts().values) == [3060, 1764, 572, 550, 177], "KMeans with n_clusters=5, n_init=10, random_state=0 inside the pipeline, then fit_predict on atlantic[ts_columns]"
assert list(pd.Series(raw_groups).value_counts().values) == [3365, 1624, 648, 370, 116], "raw_groups = kmeans.fit_predict(atlantic[ts_columns]): KMeans with no scaler, on the two columns"
print(f"✓ Your turn 2 — k-means sorted the slice twice, scaled and unscaled")

**Unscaled, k-means cuts the cloud into slices of temperature** (left), mixing fresh Antarctic and salty
North Atlantic water. **Scaled, it follows the arms** (right). From here on every method has a scaler in
front, and `groups` is kept: later sections reuse it without fitting again.

## 4. Are the groups the water masses oceanographers name?

**k-means never saw where a sample was,** so depth and latitude are a fair test: if the groups are water
masses, each should sit where the oceanographers draw one. The next cell draws them on the slice.

In [ ]:
plt.figure(figsize=(10, 4.5))
for group in range(atlantic_k):
    group_rows = atlantic[groups == group]   # the samples in this group
    plt.scatter(group_rows["lat"], group_rows["depth_m"], s=9, marker="s", label=f"group {group}")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title(f"The {atlantic_k} k-means groups on the Atlantic slice at 25.5°W")
plt.legend(loc="lower right", fontsize=8)
plt.show()

**Compare with the oceanographers' figure:**

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/lt2021_fig04.png" width="900">

*Salinity on the same line through the Atlantic (about 25°W), with the water masses named: AAIW, Antarctic Intermediate Water; MOW, Mediterranean Water; NADW, North Atlantic Deep Water; AABW, Antarctic Bottom Water. Liu and Tanhua (2021), Ocean Science, Fig. 4, CC BY 4.0.*

### ✏️ Your turn 3

**Which water mass is each group?**

**Why:** step 3, check the groups against something the method never saw: where they sit on the slice.

1. **`group_names`**: five names, group 0 first, from comparing the two figures and the layer table:
   - `"surface"`: warm surface water
   - `"central"`: central water, below the surface in the middle latitudes
   - `"AAIW"`: Antarctic Intermediate Water
   - `"NADW"`: North Atlantic Deep Water
   - `"AABW"`: Antarctic Bottom Water

   Start with the easy ones in the figure: one group lies along the top, and one on the floor in the
   south. Then find the fresh tongue near 1 km, and the salty water below it.

So: does each group sit where the oceanographers put one water mass?

**Use this name**, because the self-check looks for it: `group_names`.

In [ ]:
# ← your answer here: replace every ... with code
# the k-means figure above: group 0 is the first colour in its legend
group_names = [..., ..., ..., ..., ...]  # atlantic_k names in quotes, from the list above, group 0 first
print(group_names)

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['group_names']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
expected = ['AABW', 'NADW', 'AAIW', 'central', 'surface']
assert type(group_names) in (list, tuple), "group_names is a list: five names, each in quotes, inside [ ]"
assert ... not in group_names, "replace each ... in group_names with a name in quotes"
assert len(group_names) == 5, "give exactly five names, one per group"
unknown = [name for name in group_names if name not in expected]   # names not spelled as in the list
assert not unknown, f"{unknown}: use only names from the list above, spelled exactly as there, such as \"AAIW\""
wrong = [group for group in range(len(expected)) if group_names[group] != expected[group]]    # the groups whose name differs
assert not wrong, f"look again at group(s) {wrong}: where on the slice is it, and which water mass sits there?"
print(f"✓ Your turn 3 — the five groups are {group_names}")

The next cell draws the named groups on the slice and on the T–S diagram, and prints their medians.

In [ ]:
plt.figure(figsize=(13, 4.5))
plt.subplot(1, 2, 1)
for group in range(atlantic_k):
    group_rows = atlantic[groups == group]
    plt.scatter(group_rows["lat"], group_rows["depth_m"], s=9, marker="s", label=group_names[group])
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title(f"The {atlantic_k} k-means groups on the slice")
plt.legend(loc="lower right", fontsize=8)
plt.subplot(1, 2, 2)
for group in range(atlantic_k):
    group_rows = atlantic[groups == group]
    plt.scatter(group_rows["salinity"], group_rows["temperature_c"], s=4, label=group_names[group])
plt.xlabel("salinity")
plt.ylabel("temperature (°C)")
plt.title("The same groups on the T–S diagram")
plt.tight_layout()
plt.show()

group_medians = atlantic.groupby(groups)[["temperature_c", "salinity", "depth_m", "lat"]].median()   # the median of each column in each group
group_medians["water mass"] = group_names
print(group_medians.round(2))

**Each group sits where its water mass sits, though k-means never saw a depth.** The fresh group, AAIW
(salinity 34.47), has its middle at 800 m and
30.5°S: the intermediate layer. The salty cold group, NADW (salinity
34.92), sits at 2600 m: the deep layer. The
coldest, AABW (0.34 °C), sits at 3450 m and
50.5°S.

On the T–S diagram each group is one arm; the borders cut across the mixed water where arms join. South
of 53.5°S the coldest group fills the column from 100–400 m
down to the floor, under a thin fresh layer: the deep water there, labelled CDW (Circumpolar Deep Water) in
the figure above, is nearly as cold as the bottom water, so it shares its group.

**Mediterranean Water has no group:** all 92 of its samples (saltier than 35.3, at 800 m or deeper)
fall in the central-water group, which it resembles. Section 7 finds it another way: by where it is.

## 5. How many groups?

We told k-means to make five groups. k is a **hyperparameter**: a setting chosen before the method runs,
not learned from the data. Choose it from the question; here: **the smallest k that gives the fresh tongue,
the salty deep water and the bottom water a group each.**

### ✏️ Your turn 4

**What does k change?**

**Why:** to apply the rule above. The given loop draws the slice for k = 3 to 7 and prints each group's
median temperature, coldest first. Inside it:

1. **`kmeans_panel`**: a new model for each panel, the pipeline of Your turn 2 with `n_groups` groups (its
   own name, so `groups` of Your turn 2 stays as it is).
2. **`kmeans_labels`**: run it, `fit_predict` on `atlantic[ts_columns]`.

So: at which k do the fresh tongue, the salty deep water and the bottom water each get their own
colour, and what do the extra groups at k = 7 cut?

**Use this name**, because the self-check looks for it: `kmeans_labels`.

In [ ]:
# ← your answer here: replace every ... with code
# atlantic[ts_columns]: the slice's two columns; atlantic_k and groups: the number of groups and the groups of Your turn 2
k_values = [3, 4, 5, 6, 7]
found = []                                                               # the number of groups each panel found, for the self-check
plt.figure(figsize=(16, 8))
for i in range(len(k_values)):
    n_groups = k_values[i]                                               # k, the number of groups, for this panel
    # printed colours: 0 blue, 1 orange, 2 green, 3 red, 4 purple, 5 brown, 6 pink
    kmeans_panel = ...                                                   # a new model for this panel: the pipeline of Your turn 2, with n_groups groups
    kmeans_labels = ...                                                  # run it on the slice's two columns
    if n_groups <= atlantic_k:
        reference = groups                                               # up to k = 5: the colours of the Your turn 2 groups
    else:
        reference = previous                                             # above: the colours of the panel before
    kmeans_labels = match_colours(kmeans_labels, reference)              # match_colours, from setup: each group takes its reference group's colour               # match_colours, from setup: each group takes the colour of the Your turn 2 group it mostly holds
    previous = kmeans_labels
    medians = atlantic.groupby(kmeans_labels)["temperature_c"].median()  # the median temperature of each group
    found.append(len(medians))                                           # how many groups this panel found
    print("k =", n_groups, " colours", medians.index.values, " median °C", medians.values.round(1))
    plt.subplot(2, 3, i + 1)
    plt.scatter(atlantic["lat"], atlantic["depth_m"], c=kmeans_labels, cmap="tab10", vmin=0, vmax=9, s=4, marker="s")
    plt.gca().invert_yaxis()
    plt.xlabel("latitude (°N)")
    plt.ylabel("depth (m)")
    plt.title(f"k = {n_groups}")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['kmeans_panel', 'kmeans_labels']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert found == k_values, "each panel needs its own k: n_clusters=n_groups, not a number"
assert type(kmeans_panel) == type(make_pipeline(StandardScaler())) and len(kmeans_panel) == 2, "kmeans_panel is a pipeline: make_pipeline(StandardScaler(), KMeans(...))"
assert kmeans_panel[1].random_state == 0 and kmeans_panel[1].n_init == 10, "make_pipeline(StandardScaler(), KMeans(n_clusters=n_groups, n_init=10, random_state=0))"
assert n_groups == 7 and list(pd.Series(kmeans_labels).value_counts().values) == [2901, 1711, 523, 471, 291, 134, 92], "kmeans_labels = kmeans_panel.fit_predict(atlantic[ts_columns]), with kmeans_panel = make_pipeline(StandardScaler(), KMeans(n_clusters=n_groups, n_init=10, random_state=0))"
print(f"✓ Your turn 4 — k-means drew the slice for k = {k_values}")

**k = 3** splits the warm water into surface water (19.5 °C) and central water
(9.4 °C), but all the deep water stays one group (2.3 °C). **k = 4** splits
the deep water in two, the salty deep water (2.7 °C) and the coldest (0.5 °C),
but the fresh tongue shares the coldest group. **k = 5** first gives the fresh tongue, the salty deep
water and the bottom water a group each (the medians below 5 °C:
0.3, 2.7, 4.0). **k = 7** keeps those three and cuts the warm water into
four temperature slices (7.1, 11.7, 16.7, 23.0 °C); one reaches down into the Mediterranean
Water at 30–40°N but is mostly central water, so k = 7 adds no water mass of its own. **So k = 5.**

## 6. Does a second method find the same groups?

**A grouping is more convincing when a different method finds it too.** **Hierarchical clustering** starts
with every sample as its own group and keeps joining the two closest groups until k are left.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/ward_mechanism.png" width="760">

*Hierarchical clustering on six made-up points: the joins in order (left) and the tree they make (right). Each join sits at a height that grows with the spread it adds. A cut across the tree gives the groups. Drawn for this course.*

**Which two are closest?** Ward's answer: the two whose joining grows the spread least. The spread of a
group: each sample's distance $d$ to the group's mean, squared and added up (the same sum k-means makes small).

Example on a line, three samples at 1, 2 and 10. Join 1 and 2: mean 1.5, spread
0.5² + 0.5² = 0.5. Join 2 and 10: mean 6, spread 4² + 4² = 32. Ward joins 1
and 2 first. Close groups join early, far-apart ones last. `linkage="ward"` is this rule.

**How it differs from k-means:** k-means needs k from the start and moves k centres until each sample sits
with its nearest one. Hierarchical clustering has no centres: it joins groups step by step and records every
join in one tree, from single samples up to one group. Choosing k only cuts that tree (right panel above),
so its groups nest: k + 1 splits exactly one group in two.

### ✏️ Your turn 5

**Does hierarchical clustering find the same five water masses as k-means?**

**Why:** Your turn 2 with the second method, then a check against k-means.

1. **`hier_pipe`**: a `StandardScaler()` and hierarchical clustering with `atlantic_k` groups in a pipeline.
2. **`hier_groups`**: run it on `atlantic[ts_columns]`.

The given rest draws the two side by side, same water in the same colour, and counts the samples on which
they agree. So: do they find the same water masses?

**Use these names**, because the self-check looks for them: `hier_pipe` and `hier_groups`.

In [ ]:
# ← your answer here: replace every ... with code
# atlantic[ts_columns]: the slice's two columns; groups: the k-means groups of Your turn 2
hier_pipe = make_pipeline(..., AgglomerativeClustering(n_clusters=..., linkage="ward"))  # make the model: step 0 the scaler, step 1 hierarchical clustering with atlantic_k groups
hier_groups = hier_pipe.fit_predict(...)  # run it on the slice's two columns

# The two methods number their groups differently: match_colours (setup) gives each hierarchical group
# the number of the k-means group whose samples it mostly holds, so the same water gets the same colour.
matched = match_colours(hier_groups, groups)

plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
plt.scatter(atlantic["lat"], atlantic["depth_m"], c=groups, cmap="tab10", vmin=0, vmax=9, s=6, marker="s")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title(f"k-means, {atlantic_k} groups")
plt.subplot(1, 2, 2)
plt.scatter(atlantic["lat"], atlantic["depth_m"], c=matched, cmap="tab10", vmin=0, vmax=9, s=6, marker="s")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title(f"hierarchical clustering, {atlantic_k} groups")
plt.tight_layout()
plt.show()

print("the two methods put", (matched == groups).sum(), "of", len(groups), "samples in the same group")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['hier_pipe', 'hier_groups']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert list(pd.Series(hier_groups).value_counts().values) == [3243, 1831, 545, 385, 119], "hier_pipe = make_pipeline(StandardScaler(), AgglomerativeClustering(n_clusters=5, linkage=\"ward\")), then hier_groups = hier_pipe.fit_predict(atlantic[ts_columns])"
print(f"✓ Your turn 5 — the two methods agree on {(matched == groups).sum()} of {len(groups)} samples")

**They agree on 5,653 of 6,123 samples (92%).** Two
methods that both keep each group's spread small find nearly the same layers: the grouping does not hang on
one method.

**Think.** Where do the panels differ: inside a water mass, or at a border? At the borders: there two
water masses mix, a sample is partly both, and either answer is fair.

### ✏️ Your turn 6

**How do hierarchical clustering's groups change with k?**

**Why:** the test of Your turn 4 with the second method: its groups come from one tree, so they should
nest. The loop is the one of Your turn 4. Inside it:

1. **`hier_panel`**: a new model for each panel, the pipeline of Your turn 5 with `n_groups` groups.
   (`fit_predict` hands back only one cut of the tree, so each k is run again.)
2. **`hier_labels`**: run it on `atlantic[ts_columns]`.

So: from one k to the next, how many groups split, and do other borders move, as in Your turn 4?

**Use this name**, because the self-check looks for it: `hier_labels`.

In [ ]:
# ← your answer here: replace every ... with code
# atlantic[ts_columns]: the slice's two columns; atlantic_k and groups: the number of groups and the groups of Your turn 2
k_values = [3, 4, 5, 6, 7]
found = []                                                             # the number of groups each panel found, for the self-check
plt.figure(figsize=(16, 8))
for i in range(len(k_values)):
    n_groups = k_values[i]                                             # k, the number of groups, for this panel
    # printed colours: 0 blue, 1 orange, 2 green, 3 red, 4 purple, 5 brown, 6 pink
    hier_panel = ...                                                   # a new model for this panel: the pipeline of Your turn 5, with n_groups groups
    hier_labels = ...                                                  # run it on the slice's two columns
    if n_groups <= atlantic_k:
        reference = groups                                             # up to k = 5: the colours of the Your turn 2 groups
    else:
        reference = previous                                           # above: the colours of the panel before
    hier_labels = match_colours(hier_labels, reference)                # match_colours, from setup: each group takes its reference group's colour                   # match_colours, from setup: each group takes the colour of the Your turn 2 group it mostly holds
    previous = hier_labels
    medians = atlantic.groupby(hier_labels)["temperature_c"].median()  # the median temperature of each group
    found.append(len(medians))                                         # how many groups this panel found
    print("k =", n_groups, " colours", medians.index.values, " median °C", medians.values.round(1))
    plt.subplot(2, 3, i + 1)
    plt.scatter(atlantic["lat"], atlantic["depth_m"], c=hier_labels, cmap="tab10", vmin=0, vmax=9, s=4, marker="s")
    plt.gca().invert_yaxis()
    plt.xlabel("latitude (°N)")
    plt.ylabel("depth (m)")
    plt.title(f"hierarchical clustering, k = {n_groups}")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['hier_panel', 'hier_labels']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert found == k_values, "each panel needs its own k: n_clusters=n_groups, not a number"
assert n_groups == 7 and list(pd.Series(hier_labels).value_counts().values) == [2852, 1831, 425, 391, 385, 120, 119], "hier_labels = hier_panel.fit_predict(atlantic[ts_columns]), with hier_panel the pipeline of Your turn 5 and n_clusters=n_groups"
print(f"✓ Your turn 6 — hierarchical clustering drew the slice for k = {k_values}")

**Each step splits one group; every other border stays put.** From k = 3
to 4 the cold group splits into the bottom water (0.5 °C) and the salty deep water
(2.7 °C); from 4 to 5 the fresh tongue (3.8 °C) splits off the
bottom water. From 5 to 6 the central water splits in two, and from 6 to 7 the water between the central water and the salty deep water
(391 samples, median 800 m and 6.3 °C) splits
off: no new water mass. The groups nest; k-means moves some borders as well. By the rule of section 5,
k = 5 again.

## 7. Where does the saltiest water spread at 1000 m?

**At 30–40°N a very salty patch reaches below 1 km,** in your salinity plot (Your turn 1) and in the
oceanographers' figure (Mediterranean Water). **Where does this salty water come from?** The
Mediterranean loses more water to evaporation than rivers and rain bring, so its water is salty and
heavy. It pours out at Gibraltar, sinks, and spreads into the Atlantic near 1000 m.

The next cell draws every square at 1000 m saltier than 35.2.

In [ ]:
salty_limit = 35.2                                          # a square saltier than this counts as salty
salty = water_1000m[water_1000m["salinity"] > salty_limit]     # the salty squares at 1000 m
print(len(salty), "squares saltier than", salty_limit)

plt.figure(figsize=(9, 4.5))
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.6)
plt.scatter(salty["lon"], salty["lat"], c=salty["salinity"], cmap="viridis", vmin=salty_limit, vmax=37, s=6, marker="s")
plt.colorbar(label="salinity (the colours stop at 37)")
plt.xlim(-100, 100)
plt.ylim(-10, 70)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (°E)")
plt.ylabel("latitude (°N)")
plt.title(f"Squares saltier than {salty_limit} at 1000 m")
plt.show()

**How many separate salty patches are there, and which squares belong to none?** This question is about
place, so the method sees latitude and longitude (both in degrees: no scaler).

**DBSCAN** needs no number of groups. A square with at least `min_samples` salty squares within `eps`
degrees starts a group, which spreads to every square it can reach in such steps. So each patch, whatever
its shape, becomes one group, and a lonely square is noise, labelled -1. (k-means must be told the number of
groups, and puts every square in one.)

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/dbscan_mechanism.png" width="620">

*How DBSCAN works, on made-up points: crowded points start a group, the group spreads to every point within reach, and a lonely point is noise. Drawn for this course.*

### ✏️ Your turn 7

**Which eps finds the salty patches?**

**Why:** `eps`, how near counts as near, is DBSCAN's first hyperparameter (`min_samples` stays at 5).

1. **`eps`**: try 1, 2.5 and 10 degrees, and end on the one you keep: the one that gives each big salty area (the Atlantic
   tongue, the Mediterranean, the Arabian Sea) one group, not broken up, not joined, and no other group.
2. **`dbscan_labels`**: run the given model, `dbscan`, on `salty[["lat", "lon"]]`.

So: which eps finds the three big patches, and which squares are left as noise?

**Use these names**, because the self-check looks for them: `eps` and `dbscan_labels`.

In [ ]:
# ← your answer here: replace every ... with code
# salty: the squares saltier than salty_limit at 1000 m, from the map above
eps = ...                                # try 1, then 2.5, then 10 degrees: run this cell each time, and end on the one you keep
dbscan = DBSCAN(eps=eps, min_samples=5)  # make the model, with your eps: no scaler, both columns are in degrees
dbscan_labels = dbscan.fit_predict(...)  # run it on the squares' latitude and longitude

not_noise = dbscan_labels >= 0                                           # True for a square in a group; the rest are noise (-1)
group_sizes = pd.Series(dbscan_labels[not_noise]).value_counts().values  # the squares in each group, largest first
print(f"eps = {eps}: {len(group_sizes)} groups and {np.sum(dbscan_labels == -1)} noise squares")
print("squares in each group, largest first:", group_sizes)

plt.figure(figsize=(9, 4.5))
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.6)
plt.scatter(salty["lon"][not_noise], salty["lat"][not_noise], c=dbscan_labels[not_noise], cmap="tab20", vmin=0, vmax=19, s=6, marker="s")
plt.scatter(salty["lon"][~not_noise], salty["lat"][~not_noise], s=20, marker="x", color="black")  # ~ means not: the noise squares, as crosses
plt.xlim(-100, 100)
plt.ylim(-10, 70)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (°E)")
plt.ylabel("latitude (°N)")
plt.title(f"DBSCAN with eps = {eps}: each colour one group, crosses are noise")
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['eps', 'dbscan', 'dbscan_labels']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert dbscan.eps == eps and dbscan.min_samples == 5 and dbscan.n_features_in_ == 2, "dbscan = DBSCAN(eps=eps, min_samples=5), run on salty[[\"lat\", \"lon\"]]"
assert len(dbscan_labels) == len(salty), "dbscan_labels = dbscan.fit_predict(salty[[\"lat\", \"lon\"]]): one label per salty square"
sizes = list(pd.Series(dbscan_labels[dbscan_labels >= 0]).value_counts().values)   # the squares in each group, largest first
assert len(sizes) >= 3 and sizes[0] <= 678, "on your map, big patches share one colour: they joined; try a smaller eps"
assert sizes[:3] == [678, 243, 145], "on your map, a big patch has several colours: it broke into pieces; try a larger eps"
assert len(sizes) == 3, "a fourth, small group appeared (the Red Sea): keep the eps that gives the three big patches and nothing else; try a smaller eps"
print(f"✓ Your turn 7 — eps = {eps} finds the Atlantic tongue ({sizes[0]}), the Arabian Sea tongue ({sizes[1]}) and the Mediterranean ({sizes[2]})")

**eps = 1 gives 12 groups and 64 noise:** each patch breaks into
fragments. **eps = 10 gives 1 group.** **eps = 2.5 finds the three big patches:** the
Mediterranean (145 squares), its Atlantic tongue (678; the salty
patch of section 1), and the tongue in the Arabian Sea, fed mainly by the Red Sea (243).
Any eps from 2.5 to 6 keeps these three whole (from 3.5 the Red Sea also
forms a small group): the answer comes from the data.

**But is the Red Sea really noise?** It feeds the Arabian Sea tongue as the Mediterranean feeds the
Atlantic one; at eps = 2.5 its 7 squares are noise only because it is narrow.

### ✏️ Your turn 8

**What does min_samples change?**

**Why:** `min_samples` is how crowded a place must be to start a group: small, and a few squares make one;
large, and the thin edges of every patch become noise.

1. **`min_samples`**: try 2, 5 and 20, and end on the one you keep: the one that gives the Red Sea its own group while the three
   big patches stay whole.
2. **`dbscan_ms`** and **`ms_labels`**: a new DBSCAN with this `min_samples`, run as in Your turn 7.

So: how many salty sources and tongues are there at 1000 m?

**Use these names**, because the self-check looks for them: `min_samples` and `ms_labels`.

In [ ]:
# ← your answer here: replace every ... with code
# salty: the squares saltier than salty_limit at 1000 m; eps and dbscan_labels: the eps you kept and its groups, from Your turn 7
min_samples = ...                                    # try 2, then 5, then 20: run this cell each time, and end on the one you keep
dbscan_ms = ...                                      # a new model, with this eps and min_samples
ms_labels = ...                                      # run it on the squares' latitude and longitude, as in Your turn 7
ms_labels = match_colours(ms_labels, dbscan_labels)  # match_colours, from setup: each patch in its Your turn 7 colour

not_noise = ms_labels >= 0                                           # True for a square in a group; the rest are noise (-1)
group_sizes = pd.Series(ms_labels[not_noise]).value_counts().values  # the squares in each group, largest first
print(f"min_samples = {min_samples}: {len(group_sizes)} groups and {np.sum(ms_labels == -1)} noise squares")
print("squares in each group, largest first:", group_sizes)

plt.figure(figsize=(9, 4.5))
plt.plot(coast["lon"], coast["lat"], color="0.6", lw=0.6)
plt.scatter(salty["lon"][not_noise], salty["lat"][not_noise], c=ms_labels[not_noise], cmap="tab20", vmin=0, vmax=19, s=6, marker="s")
plt.scatter(salty["lon"][~not_noise], salty["lat"][~not_noise], s=20, marker="x", color="black")  # ~ means not: the noise squares, as crosses
plt.xlim(-100, 100)
plt.ylim(-10, 70)
plt.gca().set_aspect("equal")
plt.xlabel("longitude (°E)")
plt.ylabel("latitude (°N)")
plt.title(f"DBSCAN with eps = {eps}, min_samples = {min_samples}: each colour one group, crosses are noise")
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['min_samples', 'dbscan_ms', 'ms_labels']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert list(pd.Series(dbscan_labels[dbscan_labels >= 0]).value_counts().values) == [678, 243, 145], "Your turn 7 first: run it with the eps you kept until its self-check passes"
assert dbscan_ms.eps == eps and dbscan_ms.min_samples == min_samples and dbscan_ms.n_features_in_ == 2, "dbscan_ms = DBSCAN(eps=eps, min_samples=min_samples): your eps from Your turn 7 and this min_samples, run on salty[[\"lat\", \"lon\"]]"
assert len(ms_labels) == len(salty), "ms_labels = dbscan_ms.fit_predict(salty[[\"lat\", \"lon\"]]): one label per salty square"
sizes = list(pd.Series(ms_labels[ms_labels >= 0]).value_counts().values)   # the squares in each group, largest first
assert sizes[:3] == [678, 243, 145], "on your map a big patch broke apart or lost its edges to noise: try a smaller min_samples"
assert len(sizes) >= 4, "on your map the Red Sea is crosses, noise: try a smaller min_samples"
assert np.min(sizes) > 1, "a lone square became a group of its own: try a larger min_samples"
print(f"✓ Your turn 8 — min_samples = {min_samples} finds two salty seas and their two tongues")

**At eps = 2.5, min_samples from 2 to 4 finds 4 groups: two salty seas and the tongue each
feeds.** The Mediterranean (145 squares) feeds the Atlantic tongue (678); the
Red Sea (6 of its 7 squares at min_samples = 2; one stays noise) mainly feeds the Arabian Sea tongue (243). **5** leaves the Red Sea as noise; **20**
makes 169 squares noise and breaks the Mediterranean apart. `eps` says how near counts as
near, `min_samples` how many near squares make a patch.

### Many more clustering methods

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/sklearn_cluster_comparison.png" width="900">

*Comparing different clustering algorithms on toy datasets. Each column is one method, each row one made-up data set; the colours are the groups each method found, and black points belong to no group. scikit-learn gallery, BSD-3-Clause.*

Eleven scikit-learn methods on six made-up data sets. Find today's three: k-means (first column, a faster version called MiniBatch KMeans),
hierarchical clustering ("Ward") and DBSCAN. On round blobs (fifth row) most agree; on rings and
half-moons k-means and Ward cut across the shapes, DBSCAN follows them. The last row has no groups, yet
most methods still cut it: always check a grouping against something the method never saw.

## 8. Where was each water mass made?

**A water mass takes its temperature and salinity at the surface, then sinks,** so where a group reaches
the top of the slice is a clue to where it was made.

### ✏️ Your turn 9

**Where does each water mass reach the surface?**

**Why:** step 3 again, answering an Earth-science question. The given loop draws the top 1500 m of the
slice, one water mass at a time. Inside it:

1. **`water_mass`**: the rows of `atlantic` in this `group`.
2. **`mass_points`**: those rows as squares, latitude across and depth on the y-axis, labelled with the group's name,
   `group_names[group]` (Your turn 3).

So: which water masses reach the surface on this slice, where, and which never does?

**Use these names**, because the self-check looks for them: `water_mass` and `mass_points`.

In [ ]:
# ← your answer here: replace every ... with code
# atlantic_k, groups and group_names: the number of groups, the k-means groups and your names, from Your turns 2 and 3
plt.figure(figsize=(10, 5))
for group in range(atlantic_k):
    water_mass = ...                                                  # the rows of atlantic in this group
    mass_points = plt.scatter(..., ..., s=45, marker="s", label=...)  # latitude across, depth on the y-axis; the label is the group's name
plt.ylim(1550, -50)                                                   # only the top 1500 m, depth growing downward
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("The top 1500 m of the slice: where each water mass meets the surface")
plt.legend(loc="lower right", fontsize=8)
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['water_mass', 'mass_points']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert len(water_mass) == 177 and (water_mass.index == atlantic[groups == group].index).all(), "water_mass = atlantic[groups == group]: the rows of atlantic in this group"
assert len(mass_points.get_offsets()) == len(water_mass), "scatter the columns of water_mass, the rows of this group"
assert (mass_points.get_offsets()[:, 0] == water_mass["lat"]).all() and (mass_points.get_offsets()[:, 1] == water_mass["depth_m"]).all(), "the scatter's x is water_mass[\"lat\"] and its y water_mass[\"depth_m\"]"
drawn = mass_points.axes.collections                # every scatter of this figure, one per group
assert len(drawn) == atlantic_k, "one scatter per group: the scatter goes inside the loop"
for group in range(atlantic_k):
    assert len(drawn[group].get_offsets()) == (groups == group).sum() and drawn[group].get_label() == group_names[group], f"group {group}: water_mass = atlantic[groups == group], label=group_names[group]"
print(f"✓ Your turn 9 — drew where each water mass meets the surface")

**The North Atlantic Deep Water group reaches the surface only at the north end, between Greenland and Iceland,**
next to where it is made. **The Antarctic Intermediate Water group reaches it from Antarctica to
43.5°S;** the water sinks near the north edge of such a band and spreads north near
1000 m. **The Antarctic Bottom Water group comes up to 100 m south of
60.5°S, but never to the surface in these yearly averages:** it is made in winter under
the sea ice (B on the slide of section 1), which a yearly average hides, on the continental shelf south of
where our slice ends (the coast here is near 76°S), and sinks to the floor.

**Together the water masses are the deep half of the thermohaline circulation** (thermo, temperature;
haline, salt): water sinks in a few places, spreads along the bottom of every ocean, rises slowly and
returns along the surface.

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/thermohaline_nasa_rohde.png" width="760">

*The thermohaline circulation: surface currents in red, deep currents in blue, drawn on a map of surface salinity, with the places where deep water forms. NASA / R. Simmon, modified by R. A. Rohde (public domain).*

Deep water forms only in the far North Atlantic and around Antarctica, the two sources you found at the
ends of the slice; none in the North Pacific (the homework). Everywhere else the deep ocean holds water that
sank far away and still carries its surface temperature and salinity: that is why two numbers are enough
to sort the Atlantic's main water masses.

## Week 11 summary

### The science

**Can a computer sort the ocean into its water masses?**  
Each water mass keeps the temperature and salinity it took at the surface, so those two numbers are enough to sort the main water masses of the deep Atlantic, and where a group reaches the surface shows where it was made. Clustering takes the same steps for every method: scale and group in one pipeline, then check the groups against what the method never saw. Hyperparameters are chosen from the question: k, the smallest that gives each water mass a group; eps and min_samples, how near and how crowded a patch must be. A second method checks the first.

- **Water mass.** A body of water that took its temperature and saltiness at the surface where it sank, and carries them, slowly mixing, far from there.
- **Clustering.** Sort items into groups by their numbers alone, with no labels to learn from; whether the groups mean anything is checked afterwards.
- **Scaling factor.** How big a difference in one column counts as one step of distance; without it, the column with the biggest numbers decides everything.
- **Hyperparameter.** A setting you choose before the method runs (k, eps, min_samples): choose it from the question, and prefer a value inside a range where the answer is stable.

### The code

| Function | What it does |
|---|---|
| **pandas** | |
| `table[mask]` | keep only the rows where the mask is True — here, the squares saltier than salty_limit at 1000 m |
| **Matplotlib** | |
| `plt.scatter(x, y, c=values, cmap=...)` | colour every dot by a third column — here, the slice by temperature, by salinity, or by group |
| `plt.colorbar(label=...)` | the key that says what the colours mean — here, temperature or salinity |
| `plt.xlabel / ylabel / title / show` | label it, then draw it — every panel with its units |
| **scikit-learn** | |
| `make_pipeline(StandardScaler(), method)` | one model that scales the columns, then runs the method, as last week's SVM — here, every clustering of the week |
| `KMeans(n_clusters=k, n_init=10, random_state=0)` | k-means with k centres, started ten times, the same run for everyone — here, k = 5 for the Atlantic |
| `model.fit_predict(x)` | fit, and hand back one group number per row — the same call for every clustering method and every pipeline |
| `AgglomerativeClustering(n_clusters=k, linkage="ward")` | hierarchical clustering: keep joining the two groups whose joining adds the least spread, until k are left — here, the check on k-means |
| `DBSCAN(eps=..., min_samples=...)` | group samples that lie close together, with no k; -1 means noise — here, the salty seas and their tongues at 1000 m |

## Homework

**Is the Pacific layered like the Atlantic?** Three short slides, then four parts on the Pacific slice
at 150.5°W (`pacific`, same columns as `atlantic`): look at the data, choose k, check with a second method,
and sort both oceans with one model.
After a restart, run the setup cell; nothing else from class is needed.

**1. No deep water is made in the North Pacific.** Its surface is among the freshest on the map of
section 8 (light blue). Fresh water is light, so a sharp step in density lies a short way down, "maintained
in part by freshwater input at the sea surface" (Hanawa and Talley, 2001). Even in winter the surface water
cannot sink through it to the bottom.

**2. So the deep Pacific is filled from the south,** by deep water that enters from the Southern Ocean and
mixes slowly on its way north: **Pacific Deep Water** (the blue deep current on the map). It should have no salty layer like North Atlantic Deep Water.

**3. Two fresh waters at mid-depth.** **Antarctic Intermediate Water** from the south near 1000 m, as in
the Atlantic; and from the north a few hundred metres down, **North Pacific Intermediate Water**, made in
the Okhotsk Sea and Oyashio region east of Japan (Hanawa and Talley, 2001). Our Atlantic slice shows no
fresh northern tongue as strong.
Talley's section of our slice:

<img src="https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data/images/week11w/talley_slide_pacific_150w_salinity.png" width="820">

*Salinity in the Pacific at 150°W, the slice of this homework, with its water masses: (1) upper water, (2) AAIW from the south and NPIW from the north, (3) Pacific Deep Water, (4) the bottom water. L. Talley, SIO 210 lecture slides (2019).*

### ✏️ Your turn 10

**What do temperature and salinity look like in the Pacific?**

**Why:** as in Your turn 1, and checked against the slides.

1. **`pacific_temperature_points`** and **`pacific_salinity_points`**: the two panels of Your turn 1,
   for `pacific`.
2. **`atlantic_ts_points`** and **`pacific_ts_points`**: the T–S diagram of section 1 with both oceans on
   one figure, the Pacific on top.

So: which arm of the Atlantic's T–S cloud has no partner in the Pacific?

**Use these names**, because the self-check looks for them: `pacific_temperature_points`,
`pacific_salinity_points`, `atlantic_ts_points` and `pacific_ts_points`.

In [ ]:
# ← your answer here: replace every ... with code
# pacific: the Pacific slice; atlantic: the class slice, for comparison
plt.figure(figsize=(12, 4.5))
plt.subplot(1, 2, 1)
pacific_temperature_points = ...  # plt.scatter of pacific, as in Your turn 1, coloured (c=) by temperature; cmap="viridis", vmin=0, vmax=25, s=9, marker="s"
plt.colorbar(label="temperature (°C)")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("Temperature at 150.5°W")
plt.subplot(1, 2, 2)
pacific_salinity_points = ...     # the same scatter, coloured (c=) by salinity; cmap="viridis", vmin=34.0, vmax=35.2, s=9, marker="s"
plt.colorbar(label="salinity (the colours stop at 34.0 and 35.2)")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("Salinity at 150.5°W")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
atlantic_ts_points = ...  # plt.scatter of atlantic: salinity across, temperature up; color="0.6", s=3, label="Atlantic, 25.5°W"
pacific_ts_points = ...   # the same scatter for pacific, drawn on top; color="tab:orange", s=3, label="Pacific, 150.5°W"
plt.xlabel("salinity")
plt.ylabel("temperature (°C)")
plt.title("T–S diagrams of the two slices")
plt.legend()
plt.show()

saltiest_deep = np.max(pacific[pacific["depth_m"] >= 2000]["salinity"])  # the highest salinity below 2000 m
print("highest salinity below 2000 m in the Pacific:", round(saltiest_deep, 2))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['pacific_temperature_points', 'pacific_salinity_points', 'atlantic_ts_points', 'pacific_ts_points']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
for points in (pacific_temperature_points, pacific_salinity_points):
    assert len(points.get_offsets()) == len(pacific), "each panel draws every row of pacific: x and y are columns of pacific"
    assert points.get_array() is not None, "colour each dot by a column: c=pacific[...]"
    assert (points.get_offsets()[:, 0] == pacific["lat"]).all() and (points.get_offsets()[:, 1] == pacific["depth_m"]).all(), "in each panel, pacific[\"lat\"] goes across and pacific[\"depth_m\"] on the y-axis"
assert list(pacific_temperature_points.get_array()) == list(pacific["temperature_c"]), "the left panel: c=pacific[\"temperature_c\"]"
assert list(pacific_salinity_points.get_array()) == list(pacific["salinity"]), "the right panel: c=pacific[\"salinity\"]"
assert len(atlantic_ts_points.get_offsets()) == len(atlantic) and len(pacific_ts_points.get_offsets()) == len(pacific), "atlantic_ts_points draws atlantic's columns, pacific_ts_points pacific's"
assert (atlantic_ts_points.get_offsets()[:, 0] == atlantic["salinity"]).all() and (atlantic_ts_points.get_offsets()[:, 1] == atlantic["temperature_c"]).all(), "the Atlantic's T–S diagram: atlantic[\"salinity\"] across, atlantic[\"temperature_c\"] up"
assert (pacific_ts_points.get_offsets()[:, 0] == pacific["salinity"]).all() and (pacific_ts_points.get_offsets()[:, 1] == pacific["temperature_c"]).all(), "the Pacific's T–S diagram: pacific[\"salinity\"] across, pacific[\"temperature_c\"] up"
print(f"✓ Your turn 10 — the Pacific slice drawn by temperature and by salinity")

**What the numbers mean.** How many fresh tongues does the Pacific's salinity show, and from which side does each come? Is there a
salty deep layer (compare the printed highest salinity below 2000 m with North Atlantic Deep Water's
34.92)? Which slide explains each? Check: the northern tongue is freshest at 400 to
500 m.

### ✏️ Your turn 11

**How many groups does the Pacific need?**

**Why:** the rule of section 5: the smallest k that gives each water mass, here including North Pacific
Intermediate Water, a group of its own.

1. **`pacific_k`**: try 4, 5 and 6.
2. **`pacific_pipe`**: the pipeline of Your turn 2 with `pacific_k` groups.
3. **`pacific_groups`**: run it on `pacific[ts_columns]`.

Keep the smallest `pacific_k` with a group that is mostly the northern fresh tongue: in the printed medians, fresh,
a few hundred metres down, north of the equator. So: does the Pacific need more groups than the Atlantic?

**Use these names**, because the self-check looks for them: `pacific_k` and `pacific_groups`.

In [ ]:
# ← your answer here: replace every ... with code
# pacific: the Pacific slice; ts_columns: its temperature and salinity columns
pacific_k = ...       # try 4, then 5, then 6: run this cell each time, and end on the one you keep
pacific_pipe = ...    # make the model: the pipeline of Your turn 2, with pacific_k groups
pacific_groups = ...  # run it on the Pacific's two columns

pacific_groups = coldest_first(pacific, pacific_groups)  # coldest_first, from setup: renumber, 0 for the coldest group

plt.figure(figsize=(10, 4.5))
for group in range(pacific_k):
    group_rows = pacific[pacific_groups == group]  # the samples in this group
    plt.scatter(group_rows["lat"], group_rows["depth_m"], s=9, marker="s", label=f"group {group}")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title(f"k-means with k = {pacific_k} on the Pacific slice at 150.5°W")
plt.legend(loc="lower right", fontsize=8)
plt.show()

pacific_medians = pacific.groupby(pacific_groups)[["temperature_c", "salinity", "depth_m", "lat"]].median()  # the median of each column in each group
print(pacific_medians.round(2))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['pacific_k', 'pacific_pipe', 'pacific_groups']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert type(pacific_pipe) == type(make_pipeline(StandardScaler())) and len(pacific_pipe) == 2 and pacific_pipe[1].n_clusters == pacific_k, "pacific_pipe = make_pipeline(StandardScaler(), KMeans(n_clusters=pacific_k, ...)): the name, not a number"
assert pacific_k >= 6, "the two fresh tongues still share a group: try a larger k"
assert pacific_k == 6, "the northern fresh tongue already has its own group at a smaller k: keep the smallest k that does it"
assert list(pd.Series(pacific_groups).value_counts().values) == [4631, 1098, 269, 184, 134, 37], "make_pipeline(StandardScaler(), KMeans(n_clusters=pacific_k, n_init=10, random_state=0)).fit_predict(pacific[ts_columns])"
print(f"✓ Your turn 11 — k = {pacific_k} gives the northern fresh tongue its own group")

**What the numbers mean.** Which group is the northern fresh tongue? Give its median salinity, depth and latitude, and its name.
Why did k = 4 and k = 5 not find it? Which Atlantic water mass has no group here, and why (slide 2)?
Check: its median salinity is near 34.09.

### ✏️ Your turn 12

**How many groups does hierarchical clustering need to find North Pacific Intermediate Water?**

**Why:** a second method checks the first, with k chosen by the same rule.

1. **`pacific_hier_k`**: try 6, 7 and 8.
2. **`pacific_hier_pipe`**: the pipeline of Your turn 5 with `pacific_hier_k` groups.
3. **`pacific_hier`**: run it on `pacific[ts_columns]`.

Keep the smallest `pacific_hier_k` with a group that is mostly the northern fresh tongue (its median latitude far north). So: does it need the
same k as k-means?

**Use these names**, because the self-check looks for them: `pacific_hier_k` and `pacific_hier`.

In [ ]:
# ← your answer here: replace every ... with code
# pacific[ts_columns]: the Pacific's two columns
pacific_hier_k = ...     # try 6, then 7, then 8: run this cell each time, and end on the one you keep
pacific_hier_pipe = ...  # make the model: the pipeline of Your turn 5, with pacific_hier_k groups
pacific_hier = ...       # run it on the Pacific's two columns

pacific_hier = coldest_first(pacific, pacific_hier)  # coldest_first, from setup: renumber, 0 for the coldest group

plt.figure(figsize=(10, 4.5))
for group in range(pacific_hier_k):
    group_rows = pacific[pacific_hier == group]  # the samples in this group
    plt.scatter(group_rows["lat"], group_rows["depth_m"], s=9, marker="s", label=f"group {group}")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title(f"hierarchical clustering with k = {pacific_hier_k} on the Pacific slice")
plt.legend(loc="lower right", fontsize=8)
plt.show()

print(pacific.groupby(pacific_hier)[["temperature_c", "salinity", "depth_m", "lat"]].median().round(2))  # the median of each column in each group

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['pacific_hier_k', 'pacific_hier_pipe', 'pacific_hier']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert pacific_hier_k >= 8, "no group is mostly the northern fresh tongue yet: its freshest water still shares a group with the southern tongue's; try a larger pacific_hier_k"
assert pacific_hier_k == 8, "the northern fresh tongue already has its own group at a smaller pacific_hier_k: keep the smallest that does it"
assert list(pd.Series(pacific_hier).value_counts().values) == [4348, 644, 492, 417, 145, 144, 130, 33], "make_pipeline(StandardScaler(), AgglomerativeClustering(n_clusters=pacific_hier_k, linkage=\"ward\")).fit_predict(pacific[ts_columns])"
print(f"✓ Your turn 12 — hierarchical clustering needs k = {pacific_hier_k} to give the northern fresh tongue its own group")

**What the numbers mean.** At k = 6 and 7, where does hierarchical clustering put the freshest intermediate water: north, south, or
both? At which k is one group mostly the northern fresh tongue, and how do its medians compare with
k-means's North Pacific Intermediate Water in Your turn 11? Why can two methods need different k for the same
water mass? Check: that group's median salinity is near 34.0.

### ✏️ Your turn 13

**Which water masses do the two oceans share?**

**Why:** each ocean was sorted on its own, so their groups cannot be compared. Sorted together by one
model, the same water gets the same group in both. The given first line stacks the two tables into one,
`both` (Atlantic rows first).

1. **`both_pipe`**: a new pipeline like Your turn 11's, with `both_k` groups (given: the Pacific's k).
2. **`both_groups`**: run it on `both[ts_columns]`.

The given rest draws the two slices in the same colours and prints each group's medians and its samples in
each ocean.

So: which groups are in one ocean only, which are in both, and is each shared group really one water mass?

**Use these names**, because the self-check looks for them: `both_pipe` and `both_groups`.

In [ ]:
# ← your answer here: replace every ... with code
# pacific_k: the number of groups the Pacific needed, from Your turn 11
both = pd.concat([atlantic, pacific], ignore_index=True)  # one table: the Atlantic rows, then the Pacific rows; ignore_index numbers them again from 0
both_k = pacific_k                                        # as many groups as the Pacific needed in Your turn 11
both_pipe = ...                                           # make the model: a new pipeline like Your turn 11's, with both_k groups
both_groups = ...                                         # run it on the two columns of both

both_groups = coldest_first(both, both_groups)  # coldest_first, from setup: 0 for the coldest group

n_atlantic = len(atlantic)              # the first n_atlantic rows of both are the Atlantic
in_atlantic = both_groups[:n_atlantic]  # the groups of the Atlantic samples
in_pacific = both_groups[n_atlantic:]   # and of the Pacific samples

plt.figure(figsize=(13, 4.5))
plt.subplot(1, 2, 1)
for group in range(both_k):
    group_rows = atlantic[in_atlantic == group]
    plt.scatter(group_rows["lat"], group_rows["depth_m"], s=6, marker="s", label=f"group {group}")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("The Atlantic at 25.5°W")
plt.legend(loc="lower right", fontsize=7)
plt.subplot(1, 2, 2)
for group in range(both_k):
    group_rows = pacific[in_pacific == group]
    plt.scatter(group_rows["lat"], group_rows["depth_m"], s=6, marker="s", label=f"group {group}")
plt.gca().invert_yaxis()
plt.xlabel("latitude (°N)")
plt.ylabel("depth (m)")
plt.title("The Pacific at 150.5°W, the same groups")
plt.legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.show()

summary = both.groupby(both_groups)[["temperature_c", "salinity", "depth_m"]].median().round(2)  # each group's medians
atlantic_count = []
pacific_count = []
for group in range(both_k):
    atlantic_count.append(np.sum(in_atlantic == group))  # Atlantic samples in this group
    pacific_count.append(np.sum(in_pacific == group))    # Pacific samples in this group
summary["Atlantic samples"] = atlantic_count
summary["Pacific samples"] = pacific_count
print(summary)

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
for name in ['both_pipe', 'both_groups']:
    assert name in globals() and globals()[name] is not ..., f"fill in {name} in the cell above and run it first"
assert both_k == 6, "both_k comes from pacific_k: go back to Your turn 11 and let its self-check pass"
assert len(both_groups) == len(atlantic) + len(pacific), "both_groups = both_pipe.fit_predict(both[ts_columns]): one group for every row of both"
assert list(pd.Series(both_groups).value_counts().values) == [6716, 3078, 1727, 665, 250, 40], "both_pipe = make_pipeline(StandardScaler(), KMeans(n_clusters=both_k, n_init=10, random_state=0))"
print(f"✓ Your turn 13 — sorted both oceans with one model")

**What the numbers mean.** Which group is in the Atlantic only, and which in the Pacific only? Which group holds both the Atlantic's
bottom water and the deep Pacific, and is that proof of where the deep Pacific comes from? Where does the
shared fresh group sit in the Pacific panel, and what does that say about using k = 6 for two oceans?
Check: one group has no Pacific samples at all.